In [1]:
import ee
import geemap

ee.Initialize(project='flooding-map-project')

print("GEE pronto.")

GEE pronto.


In [2]:
#Zona Ravenna

aoi = ee.Geometry.Rectangle([11.8, 44.2, 12.4, 44.6])

print("AOI definita:", aoi.getInfo()['type'])

AOI definita: Polygon


In [3]:
# Immagini Sentinel-1 PRE-alluvione (aprile 2023)
s1_pre = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2023-04-01', '2023-04-30')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select('VV')  # prendiamo solo la banda VV
)

print('Immagini pre-alluvione trovate:', s1_pre.size().getInfo())

Immagini pre-alluvione trovate: 3


In [4]:
# Immagini Sentinel-1 POST-alluvione (maggio 2023)
s1_post = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate('2023-05-15', '2023-05-31')
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING'))
    .select('VV')
)

print('Immagini post-alluvione trovate:', s1_post.size().getInfo())

Immagini post-alluvione trovate: 1


In [5]:
# Invece di usare una singola immagine (che potrebbe avere anomalie), faccio la media di tutte le immagini disponibili per ciascun periodo. Ciò riduce il rumore e lo speckle in modo naturale



# Media di tutte le immagini pre-alluvione
img_pre = s1_pre.mean().clip(aoi)

# Media di tutte le immagini post-alluvione  
img_post = s1_post.mean().clip(aoi)

print("Immagine pre creata — bande:", img_pre.bandNames().getInfo())
print("Immagine post creata — bande:", img_post.bandNames().getInfo())

Immagine pre creata — bande: ['VV']
Immagine post creata — bande: ['VV']


In [6]:
# Parametri di visualizzazione per immagini SAR in dB
# I valori tipici di backscatter VV vanno da -25 dB (acqua) a -5 dB (urbano)
vis_params = {
    'min': -25,
    'max': 0,
    'palette': ['black', 'white']
}

# Creiamo la mappa interattiva
Map = geemap.Map(center=[44.3, 12.0], zoom=9)

Map.addLayer(img_pre,  vis_params, 'Pre-alluvione (aprile 2023)')
Map.addLayer(img_post, vis_params, 'Post-alluvione (maggio 2023)')

Map.addLayerControl()
Map

Map(center=[44.3, 12.0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

In [7]:
import os

project_root = os.path.dirname(os.getcwd())
output_dir = os.path.join(project_root, 'data', 'raw')
os.makedirs(output_dir, exist_ok=True)

# Esporto img_pre come GeoTIFF
geemap.ee_export_image(
    img_pre,
    filename=os.path.join(output_dir, 's1_pre_april2023.tif'),
    scale=30,          # risoluzione 30 metri per pixel
    region=aoi,
    file_per_band=False
)

print("Salvataggio in:", output_dir)

Generating URL ...
Please wait ...
Data downloaded to c:\Users\frate\Desktop\FloodingMap_project\data\raw\s1_pre_april2023.tif
Salvataggio in: c:\Users\frate\Desktop\FloodingMap_project\data\raw


In [8]:
geemap.ee_export_image(
    img_post,
    filename=os.path.join(output_dir, 's1_post_may2023.tif'),
    scale=30,
    region=aoi,
    file_per_band=False
)
print("Salvataggio in:", output_dir)

Generating URL ...
Please wait ...
Data downloaded to c:\Users\frate\Desktop\FloodingMap_project\data\raw\s1_post_may2023.tif
Salvataggio in: c:\Users\frate\Desktop\FloodingMap_project\data\raw
